# 🔴 미션 3 (선택) — 캡스톤: 옵션을 골라 결과를 만든다

오늘 배운 옵션으로 **내가 쓸 만하다고 판단한 결과물 하나**를 만든다.
아래 **A · B 중 하나**를 고른다.

| | 무엇을 | 쓰는 모델 |
|---|---|---|
| **A 이어쓰기** | 내 첫 문장 → 이야기 이어쓰기 | KoGPT2 (오늘) |
| **B 요약 다듬기** | 내 글 → 요약. 옵션을 바꿔 제일 나은 요약 고르기 | KoBART (Day 4) |

> A가 기본이다. B는 Day 4 요약을 다시 꺼내 오늘 배운 옵션을 붙여 보는 쪽이다.

## 낼 것

결과 1개 + **입력 · 사용한 옵션 · 출력**을 적은 표

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device :", device)

---
# A. 이어쓰기

A를 고르면 아래를 실행한다. (B를 고르면 이 절을 건너뛴다)

In [ ]:
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel

gpt_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    "skt/kogpt2-base-v2",
    bos_token="</s>", eos_token="</s>",
    unk_token="<unk>", pad_token="<pad>", mask_token="<mask>",
)
gpt_model = GPT2LMHeadModel.from_pretrained("skt/kogpt2-base-v2").eval().to(device)


def 이어쓰기(prompt, **options):
    ids = gpt_tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = gpt_model.generate(ids, max_new_tokens=120,
                                 pad_token_id=gpt_tokenizer.pad_token_id, **options)
    return gpt_tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
내_첫문장 = "그날 아침, 우체통에는 낯선 편지가 한 통 들어 있었다."   # ← 내 문장으로

# 옵션을 바꿔 가며 세 번 뽑고, 제일 나은 것을 고른다
내_옵션들 = [
    dict(do_sample=True, top_p=0.92, top_k=0, repetition_penalty=1.1),
    dict(do_sample=True, temperature=0.8, top_k=50, repetition_penalty=1.2),
    dict(do_sample=True, top_p=0.95, top_k=0, repetition_penalty=1.3),
]

for n, options in enumerate(내_옵션들, start=1):
    torch.manual_seed(n)
    print("=" * 60)
    print(f"[{n}] {options}")
    print(이어쓰기(내_첫문장, **options))
    print()

---
# B. 요약 다듬기

B를 고르면 아래를 실행한다. Day 4에서 쓴 KoBART 를 다시 쓴다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

bart_tokenizer = AutoTokenizer.from_pretrained("gogamza/kobart-base-v2")
bart_model = AutoModelForSeq2SeqLM.from_pretrained(
    "gogamza/kobart-summarization").eval().to(device)


def 요약(text, **options):
    ids = bart_tokenizer.encode(text, return_tensors="pt",
                                max_length=1024, truncation=True).to(device)
    with torch.no_grad():
        out = bart_model.generate(ids, max_new_tokens=100,
                                  pad_token_id=bart_tokenizer.pad_token_id, **options)
    return bart_tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
# ← 아래 예시를 내 글(뉴스·블로그 등 여러 문단짜리)로 바꾼다. 짧으면 요약할 것이 없다.
내_글 = """
정부가 인공지능 산업 육성을 위한 종합 대책을 발표했다.
이번 대책에는 국내 기업의 인공지능 모델 개발을 지원하기 위한 대규모 연산 자원 확충 계획이 담겼다.
정부는 향후 3년간 공공 데이터센터에 고성능 연산 장비를 순차적으로 도입하고,
중소기업과 연구기관이 이를 저렴하게 사용할 수 있도록 이용료를 지원하기로 했다.

인재 양성 방안도 포함됐다. 대학과 협력해 인공지능 전공 과정을 확대하고,
재직자를 대상으로 한 재교육 과정을 신설한다. 특히 비전공자가 인공지능 실무 역량을
갖출 수 있도록 하는 단기 집중 과정을 늘리겠다는 계획이다.

업계에서는 연산 자원 지원을 반기면서도, 실제 집행 속도가 관건이라는 반응이 나온다.
한 관계자는 "장비 도입보다 이를 실제로 쓸 수 있게 하는 운영 인력과 절차가 더 중요하다"고 말했다.
""".strip()

내_옵션들 = [
    dict(do_sample=False),                                    # greedy
    dict(do_sample=False, num_beams=4),                       # beam
    dict(do_sample=False, num_beams=4, repetition_penalty=1.2),
]

for n, options in enumerate(내_옵션들, start=1):
    print("=" * 60)
    print(f"[{n}] {options}")
    print(요약(내_글, **options))
    print()

---
## 제출

고른 결과 하나를 아래 표로 정리해서 낸다.

```
고른 쪽 : A 이어쓰기 / B 요약 다듬기

입력 :

사용한 옵션 :          (예: do_sample=True, top_p=0.92, repetition_penalty=1.1)

출력 :

이걸 고른 이유 한 줄 :
```

> **옵션을 반드시 적는다.** 같은 모델도 옵션에 따라 전혀 다른 결과가 나온다는 것이
> 오늘의 요점이므로, 결과만 있고 옵션이 없으면 재현할 수 없다.

## 🔵 더 해 보고 싶다면 — 정해진 답 없음

- **모델에게 사실을 물어본다** — `"대한민국의 수도는"` · `"세종대왕이 만든 것은"`.
  맞히나? 틀린다면 **어떤 식으로** 틀리는가? (수업 `## 6` 환각 이야기와 이어진다)
- **내가 첫 문장을 쓰고 모델이 이어 쓴다.** 그걸 다시 내가 고쳐 쓰고 또 이어 붙인다.
  번갈아 가며 짧은 이야기 하나를 끝까지 만들어 본다.
- 같은 프롬프트를 **로컬 모델과 API 모델**에 각각 넣고 나란히 놓아 본다.
  125M과 상용 모델의 차이가 어디에서 제일 크게 보이나?
- 오늘 배운 옵션으로 **제일 마음에 드는 문장 하나**를 뽑아 본다. 몇 번 만에 나왔나?